In [1]:
import pandas as pd
import torch
import numpy as np
import sklearn
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.metrics import accuracy_score


print(torch.cuda.is_available())  
print(torch.cuda.device_count())  
print(torch.cuda.get_device_name(0)) 

True
1
NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [2]:
# for a two-class tree, call this function like this:
# writegraphtofile(clf, ('F', 'T'), dirname+graphfilename)
# for a multi-class tree, call this function like this:
# @ writegraphtofile(clf, featurenames, dirname+graphfilename)
def writegraphtofile(clf, featurelabels, filename):
 dot_data = sklearn.tree.export_graphviz(clf, feature_names=featurelabels, out_file=None)
 graph=pydotplus.graph_from_dot_data(dot_data)
 graph.write_png(filename)

In [3]:
pd.set_option('display.max_columns', None)

## Load Dataset 

In [4]:
file_path = r"/home/kobugi/papaya/ece-5464/project_2/diabetic_data.csv"

In [5]:
df = pd.read_csv(file_path)
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0.0,1.0,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,,59,0.0,18.0,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,,11,5.0,13.0,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,,44,1.0,16.0,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,,51,0.0,8.0,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [6]:
df.shape


(101766, 50)

## Check for Data Errors & Detect and correct missing values

In [7]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing values per column:\n", missing_values)


Missing values per column:
 encounter_id                    0
patient_nbr                     0
race                            0
gender                          0
age                             0
weight                          0
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                      0
medical_specialty               0
num_lab_procedures              0
num_procedures                 43
num_medications                 6
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                          0
diag_2                          0
diag_3                          0
number_diagnoses                0
max_glu_serum               96420
A1Cresult                   84748
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride         

In [8]:
print(df["num_procedures"].median(),df["num_medications"].median())

1.0 15.0


In [9]:
# median value fillna was helped by ChatGPT 4o
median_value = df["num_procedures"].median()
medi_median_value = df["num_medications"].median()
df.fillna({"num_procedures":median_value}, inplace=True)
df.fillna({"num_medications":median_value}, inplace=True)

In [10]:
# check if two columns' missing values been filled.
missing_values = df.isnull().sum()
print("Missing values per column:\n", missing_values)

Missing values per column:
 encounter_id                    0
patient_nbr                     0
race                            0
gender                          0
age                             0
weight                          0
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                      0
medical_specialty               0
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                          0
diag_2                          0
diag_3                          0
number_diagnoses                0
max_glu_serum               96420
A1Cresult                   84748
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride         

In [11]:
# Check for "?" in the entire DataFrame
question_marks = df.applymap(lambda x: x == "?")

# Sum up how many "?" values are in each column
question_mark_count = question_marks.sum()

print("Number of '?' values per column:\n", question_mark_count)


/tmp/ipykernel_40806/2145558370.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  question_marks = df.applymap(lambda x: x == "?")


Number of '?' values per column:
 encounter_id                    0
patient_nbr                     0
race                         2273
gender                          0
age                             0
weight                      98569
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                  40256
medical_specialty               0
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                         21
diag_2                        358
diag_3                       1423
number_diagnoses                0
max_glu_serum                   0
A1Cresult                       0
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride   

In [12]:
# consulted with ChatGPT 4o on imputing method and code. Asked for whether to use kNN or Mode, and the answer was mix of both.
from sklearn.impute import KNNImputer
import pandas as pd

# Replace '?' with NaN
columns_to_replace = ["race", "payer_code", "diag_1", "diag_2", "diag_3"]
df[columns_to_replace] = df[columns_to_replace].replace("?", pd.NA)

# Fill categorical columns with mode (updated method)
df["race"] = df["race"].fillna(df["race"].mode()[0])
df["payer_code"] = df["payer_code"].fillna(df["payer_code"].mode()[0])

# Convert diagnosis codes to numeric (for kNN), keeping NaNs
for col in ["diag_1", "diag_2", "diag_3"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Apply kNN Imputation for diagnosis columns
imputer = KNNImputer(n_neighbors=5)  # Using 5 nearest neighbors
df[["diag_1", "diag_2", "diag_3"]] = imputer.fit_transform(df[["diag_1", "diag_2", "diag_3"]])

# Confirm missing values are handled
print(df[["race", "payer_code", "diag_1", "diag_2", "diag_3"]].isna().sum())


race          0
payer_code    0
diag_1        0
diag_2        0
diag_3        0
dtype: int64


# Questions
- Payer code does not seem to mean anything, and has over 30 % missing values. Okay to delete it? 

In [13]:
# drop three columns that has over 50% missing values. "max_glu_serum","A1Cresult", "weight"
# drop meaningless column that has about 30 % missing values "payer_code"
df = df.drop(["max_glu_serum","A1Cresult", "weight","payer_code",], axis=1)

In [14]:
df.head()

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,Pediatrics-Endocrinology,41,0.0,1.0,0,0,0,250.83,565.00,477.72,1,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,,59,0.0,18.0,0,0,0,276.00,250.01,255.00,9,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,,11,5.0,13.0,2,0,1,648.00,250.00,616.80,6,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,,44,1.0,16.0,0,0,0,8.00,250.43,403.00,7,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,,51,0.0,8.0,0,0,0,197.00,157.00,250.00,5,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [15]:

# factorize age for ordinal values
df["age"]= df["age"].factorize()[0]

# separate target values before one hot encoding

x_df = df.drop(["readmitted"], axis=1)
y = df["readmitted"]
# do one hot encoding for categorical
x_df = pd.get_dummies(x_df)

In [16]:
x_df.head()

,encounter_id,patient_nbr,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,race_AfricanAmerican,race_Asian,race_Caucasian,race_Hispanic,race_Other,gender_Female,gender_Male,gender_Unknown/Invalid,medical_specialty_,medical_specialty_AllergyandImmunology,medical_specialty_Anesthesiology,medical_specialty_Anesthesiology-Pediatric,medical_specialty_Cardiology,medical_specialty_Cardiology-Pediatric,medical_specialty_DCPTEAM,medical_specialty_Dentistry,medical_specialty_Dermatology,medical_specialty_Emergency/Trauma,medical_specialty_Endocrinology,medical_specialty_Endocrinology-Metabolism,medical_specialty_Family/GeneralPractice,medical_specialty_Gastroenterology,medical_specialty_Gynecology,medical_specialty_Hematology,medical_specialty_Hematology/Oncology,medical_specialty_Hospitalist,medical_specialty_InfectiousDiseases,medical_specialty_InternalMedicine,medical_specialty_Nephrology,medical_specialty_Neurology,medical_specialty_Neurophysiology,medical_specialty_Obsterics&Gynecology-GynecologicOnco,medical_specialty_Obstetrics,medical_specialty_ObstetricsandGynecology,medical_specialty_Oncology,medical_specialty_Ophthalmology,medical_specialty_Orthopedics,medical_specialty_Orthopedics-Reconstructive,medical_specialty_Osteopath,medical_specialty_Otolaryngology,medical_specialty_OutreachServices,medical_specialty_Pathology,medical_specialty_Pediatrics,medical_specialty_Pediatrics-AllergyandImmunology,medical_specialty_Pediatrics-CriticalCare,medical_specialty_Pediatrics-EmergencyMedicine,medical_specialty_Pediatrics-Endocrinology,medical_specialty_Pediatrics-Hematology-Oncology,medical_specialty_Pediatrics-InfectiousDiseases,medical_specialty_Pediatrics-Neurology,medical_specialty_Pediatrics-Pulmonology,medical_specialty_Perinatology,medical_specialty_PhysicalMedicineandRehabilitation,medical_specialty_PhysicianNotFound,medical_specialty_Podiatry,medical_specialty_Proctology,medical_specialty_Psychiatry,medical_specialty_Psychiatry-Addictive,medical_specialty_Psychiatry-Child/Adolescent,medical_specialty_Psychology,medical_specialty_Pulmonology,medical_specialty_Radiologist,medical_specialty_Radiology,medical_specialty_Resident,medical_specialty_Rheumatology,medical_specialty_Speech,medical_specialty_SportsMedicine,medical_specialty_Surgeon,medical_specialty_Surgery-Cardiovascular,medical_specialty_Surgery-Cardiovascular/Thoracic,medical_specialty_Surgery-Colon&Rectal,medical_specialty_Surgery-General,medical_specialty_Surgery-Maxillofacial,medical_specialty_Surgery-Neuro,medical_specialty_Surgery-Pediatric,medical_specialty_Surgery-Plastic,medical_specialty_Surgery-PlasticwithinHeadandNeck,medical_specialty_Surgery-Thoracic,medical_specialty_Surgery-Vascular,medical_specialty_SurgicalSpecialty,medical_specialty_Urology,metformin_Down,metformin_No,metformin_Steady,metformin_Up,repaglinide_Down,repaglinide_No,repaglinide_Steady,repaglinide_Up,nateglinide_Down,nateglinide_No,nateglinide_Steady,nateglinide_Up,chlorpropamide_Down,chlorpropamide_No,chlorpropamide_Steady,chlorpropamide_Up,glimepiride_Down,glimepiride_No,glimepiride_Steady,glimepiride_Up,acetohexamide_No,acetohexamide_Steady,glipizide_Down,glipizide_No,glipizide_Steady,glipizide_Up,glyburide_Down,glyburide_No,glyburide_Steady,glyburide_Up,tolbutamide_No,tolbutamide_Steady,pioglitazone_Down,pioglitazone_No,pioglitazone_Steady,pioglitazone_Up,rosiglitazone_Down,rosiglitazone_No,rosiglitazone_Steady,rosiglitazone_Up,acarbose_Down,acarbose_No,acarbose_Steady,acarbose_Up,miglitol_Down,miglitol_No,miglitol_Steady,miglitol_Up,troglitazone_No,troglitazone_Steady,tolazamide_No,tolazamide_Steady,tolazamide_Up,examide_No,citoglipton_No,insulin_Down,insulin_No,insulin_Steady,insulin_Up,glyburide-metformin_Down,glyburide-metformin_No,glyburide-metformin_Steady,glyburide-metformin_Up,glipizide-metformin_No,glipiz

### Questions
- part 3 is data preparation with Python and pandas right? No using excel? 
- What to do with missing values?
- if it is empty.. num_procedures.. numeric .. - 0, max_glu_serum categorical, A1Cresult categorical..
- depending on type of data.. lecture 6 slide 19 ..
- 
- if it is '?'.. race, weight, payer_code, diag_1, diag_2, diag_3
- Is this below pd.get_dummies(df) right way to deal with categorical? 


In [17]:
# Check the data types of each column
print(df.dtypes)

encounter_id                  int64
patient_nbr                   int64
race                         object
gender                       object
age                           int64
admission_type_id             int64
discharge_disposition_id      int64
admission_source_id           int64
time_in_hospital              int64
medical_specialty            object
num_lab_procedures            int64
num_procedures              float64
num_medications             float64
number_outpatient             int64
number_emergency              int64
number_inpatient              int64
diag_1                      float64
diag_2                      float64
diag_3                      float64
number_diagnoses              int64
metformin                    object
repaglinide                  object
nateglinide                  object
chlorpropamide               object
glimepiride                  object
acetohexamide                object
glipizide                    object
glyburide                   

In [18]:
from sklearn.model_selection import train_test_split

# Define features (X) and target variable (y)
X = x_df
# Target variable.. y already set before

# First split: 60% training, 40% remaining
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=28)

# Second split: 50% of remaining (i.e., 20% of total) for validation, 50% for test
#X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Print dataset sizes to confirm
print(f"Training set: {len(X_train)} samples")
#print(f"Validation set: {len(X_val)} samples")
print(f"Test set: {len(X_test)} samples")

Training set: 71236 samples
Test set: 30530 samples


In [20]:
clf = DecisionTreeClassifier(criterion='entropy', random_state=31,max_depth =4)

In [21]:
clf.fit(X_train, y_train)

DecisionTreeClassifier(criterion='entropy', max_depth=4, random_state=31)

In [22]:
y_pred = clf.predict(X_test)

In [23]:
clf.score(X_test,y_test)

0.576842450049132

In [24]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Multivariate model accuracy: {accuracy:.2f}")

Multivariate model accuracy: 0.58


In [25]:
feature_labels = x_df.columns

In [26]:
writegraphtofile(clf,feature_labels, 'multi_decision_tree.png')

NameError: name 'pydotplus' is not defined

## Test for Binary Model

In [ ]:
# Replace all occurrences of 'B' in 'col2' with 'Y'
df['readmitted'].replace('>30', 'YES', inplace=True)
df['readmitted'].replace('<30', 'YES', inplace = True)
print(df['readmitted'].head(50))

In [ ]:
# Define features (X) and target variable (y)
X = x_df
# Target variable.. y already set before
y = df['readmitted']
# First split: 60% training, 40% remaining
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=28)
# Print dataset sizes to confirm
print(f"Training set: {len(X_train)} samples")
#print(f"Validation set: {len(X_val)} samples")
print(f"Test set: {len(X_test)} samples")


In [ ]:
clf = DecisionTreeClassifier(criterion='entropy', random_state=31,max_depth =4)

In [ ]:
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)

In [ ]:
clf.score(X_test,y_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Binary model accuracy: {accuracy:.2f}")

In [ ]:
writegraphtofile(clf,feature_labels, 'binary_decision_tree.png')